# DistrictProblem + NSGA-II: исследовательские варианты

Утверждённые сценарии не заменяются. 64 особи, 100 поколений, seeds42/43/44; цели — стоимость земли с окружением и NPV. Кампус физически невыполним в заданных границах; школа без проектных зданий — блокировка входными данными, не нулевая стоимость. Для марины и детсада сохраняются фронты и повторно проверенные варианты. Кэш расчёта привязан к хешам модели, кода и входов.

In [1]:
from pathlib import Path
import csv, json, subprocess
ROOT=Path.cwd()
if ROOT.name=='notebooks': ROOT=ROOT.parent
assert (ROOT/'scripts/land_value.py').is_file()


In [2]:
result=subprocess.run([str(ROOT/'.venv-land/bin/python'),str(ROOT/'scripts/district_optimization.py')],cwd=ROOT,capture_output=True,text=True,check=True)
print(result.stdout[-5000:])


PROJECT campus quarter 157
RESUME campus 42
RESUME campus 43
RESUME campus 44
PROJECT marina quarter 227
RESUME marina 42
RESUME marina 43
RESUME marina 44
PROJECT sailing_school quarter 205
PROJECT kindergarten quarter 232
RESUME kindergarten 42
RESUME kindergarten 43
RESUME kindergarten 44



In [3]:
runs=list(csv.DictReader((ROOT/'reports/optimization_runs.csv').open()))
assert len(runs)==12
executed=[r for r in runs if int(r['evaluations'])>0]
blocked=[r for r in runs if r['status']=='blocked_missing_project_geometry']
assert len(executed)==9 and len(blocked)==3
assert sum(int(r['evaluations']) for r in executed)==57600
assert all(int(r['generations_executed'])==100 for r in executed)
assert all(r['status']=='infeasible' for r in runs if r['project']=='campus')
selected=list(csv.DictReader((ROOT/'reports/optimization_selected.csv').open()))
assert len(selected)==6
assert all(float(r['max_constraint'])<=1e-8 for r in selected)
for row in selected:
    assert abs(float(row['recalculated_land_rub'])-float(row['land_value_total_rub']))<.1
    assert abs(float(row['recalculated_npv_rub'])-float(row['investor_npv_rub']))<.1
print('9 полных запусков, 57 600 оценок; 3 запуска школы заблокированы входными данными.')
print('6 выбранных вариантов марины/детсада перепроверены прямым расчётом.')
for row in runs: print(row['project'],row['seed'],row['status'],row['evaluations'])


9 полных запусков, 57 600 оценок; 3 запуска школы заблокированы входными данными.
6 выбранных вариантов марины/детсада перепроверены прямым расчётом.
campus 42 infeasible 6400
campus 43 infeasible 6400
campus 44 infeasible 6400
marina 42 completed 6400
marina 43 completed 6400
marina 44 completed 6400
sailing_school 42 blocked_missing_project_geometry 0
sailing_school 43 blocked_missing_project_geometry 0
sailing_school 44 blocked_missing_project_geometry 0
kindergarten 42 completed 6400
kindergarten 43 completed 6400
kindergarten 44 completed 6400
